# Jahresergebnisse von Gebäudetypologien auswerten

Dieses How-to lädt die Jahreswerte aus einem vorhandenen Typologieordner.
Jede Zeile steht für ein simuliertes Szenario. Installiere im Repository
`python -m pip install -e ".[analysis]"` und wähle dieselbe Umgebung als Kernel.

Die Erzeugung der Parquet-Dateien aus annual-Excel-Dateien und das Zusammenführen
mehrerer Läufe zeigt [Simulationsläufe mit Parquet analysieren](Gebäudesektormodell/building_sector_analysis.ipynb).

In [1]:
#| code-fold: true
#| code-summary: "Vorbereitung für lokale Ausführung"
from pathlib import Path

import pandas as pd
import plotly.io as pio
import pyarrow.parquet as pq
from IPython.display import display

from pexl import plot, reporting
from pexl.plot.reports import render

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "src" / "pexl").is_dir()), Path.cwd())
pio.renderers.default = "plotly_mimetype"

## Typologieordner wählen

Setze `TYPOLOGY_DIR` auf einen absoluten oder zum Repository relativen Pfad.
Das Notebook lädt standardmäßig `scenarios.parquet`. Ist nur eine anders
benannte Parquet-Datei vorhanden, verwendet es diese. Bei mehreren möglichen
Dateien kannst du den Namen über `PARQUET_NAME` angeben.

In [2]:
TYPOLOGY_DIR = Path("data/building_typologies/building_sector_v2")
PARQUET_NAME = None  # Optional, z. B. "scenarios_harmonisiert.parquet"
LANGUAGE = "de"

## Szenarien und Kennzahlen laden

Für die Übersicht werden nur die benötigten Spalten geladen. Der Index aus
`batch` und `scenario_name` unterscheidet gleichnamige Szenarien aus mehreren Läufen.

In [3]:
directory = ROOT / TYPOLOGY_DIR
parquet_path = directory / (PARQUET_NAME or "scenarios.parquet")
if PARQUET_NAME is None and not parquet_path.is_file():
    candidates = sorted(directory.glob("*.parquet"))
    if not candidates:
        raise FileNotFoundError(f"Keine Parquet-Datei in {directory} gefunden.")
    if len(candidates) > 1:
        raise ValueError(f"Mehrere Parquet-Dateien in {directory}. Setze PARQUET_NAME.")
    parquet_path = candidates[0]

identity = ["batch", "scenario_name"]
geometry = "preset_recorded_geometry"
envelope = "preset_recorded_hulls"
heating = "preset_recorded_heating_system"
required = [*identity, geometry, envelope, "QH"]
optional = ["QC", "FSI", "PEI_balance", "PEI_virtual_demand", "PEI_virtual_supply",
            "PV_roof_share", "weather_name"]
available = pq.ParquetFile(parquet_path).schema_arrow.names
missing = [name for name in required if name not in available]
if missing:
    raise ValueError(f"Fehlende Spalten in {parquet_path.name}: {missing}")

presets = [name for name in available if name.startswith("preset_recorded_")]
columns = list(dict.fromkeys(required + presets + [name for name in optional if name in available]))
scenarios = pd.read_parquet(parquet_path, columns=columns).set_index(identity, verify_integrity=True)
print(f"{directory.name}/{parquet_path.name}: {len(scenarios):,} Szenarien")
display(scenarios.groupby(level="batch").size().rename("Szenarien"))
display(scenarios[[geometry, envelope, "QH"]].head())

building_sector_v2/scenarios.parquet: 32,400 Szenarien


batch
building_sector_v2    32400
Name: Szenarien, dtype: int64

preset_recorded_geometry  ...          QH
batch              scenario_name                                                                ...            
building_sector_v2 EFH_1G|vor1919|WP_Sole|PV30|noFlex|Wien |Leicht                      EFH_1G  ...  481.640632
                   EFH_1G|vor1919|WP_Sole|PV30|noFlex|Wien |mittel                      EFH_1G  ...  481.553227
                   EFH_1G|vor1919|WP_Sole|PV30|noFlex|Wien |schwer                      EFH_1G  ...  481.637805
                   EFH_1G|vor1919|WP_Sole|PV30|noFlex|Innsbruck |L...                   EFH_1G  ...  509.486503
                   EFH_1G|vor1919|WP_Sole|PV30|noFlex|Innsbruck |m...                   EFH_1G  ...  509.137928

[5 rows x 3 columns]

## Presets erkunden und Varianten auswählen

Standardmäßig wird der häufigste Geometrietyp ausgewählt. Ersetze die Werte
in `filters` für deine Fragestellung. Mehrere Werte derselben Spalte werden
mit ODER verknüpft, mehrere Spalten mit UND.

In [4]:
for name in (geometry, envelope, heating):
    if name in scenarios:
        display(scenarios[name].value_counts(dropna=False).rename("Szenarien").to_frame())

filters = {geometry: scenarios[geometry].value_counts().head(1).index.tolist()}
# filters[envelope] = ["…"]
# filters[heating] = ["…"]

selected = scenarios
for name, values in filters.items():
    if values:
        selected = selected.loc[selected[name].isin(values)]

if selected.empty:
    raise ValueError("Die Filterkombination hat keine Treffer. Passe filters an.")
print(f"{len(selected):,} Szenarien in der Auswahl")

3,240 Szenarien in der Auswahl


,Szenarien
preset_recorded_geometry,
EFH_1G,3240
GWB_7G,3240
EFH_2G,3240
NiWo_2G,3240
NiWo_3G,3240
RH_2G,3240
MFH_2G,3240
MFH_4G,3240
NiWo_1G,3240


,Szenarien
preset_recorded_hulls,
MFH_bis1919,3600
MFH_1919 - 44,3600
MFH_1945 - 59,3600
MFH_1960 - 79,3600
MFH_1980 - 89,3600
MFH_1990 - 99,3600
"WD1 (1/3, 10cm)",3600
"WD2 (2/3, 20cm)",3600
"WD3 (3/3, 30cm)",3600


,Szenarien
preset_recorded_heating_system,
Wärmepumpe (Erdwärme),6480
Wärmepumpe (Grundwasser),3240
Wärmepumpe (Luft-Wasser),3240
Fernwärme Wien,3240
Fernwärme (Heizwerk erneuerbar),3240
Biomasse,3240
Erdgasheizung,3240
Ölheizung,3240
Elektroheizung,3240


## Verteilungen und Primärenergie

Die Verteilungen beschreiben ungewichtete Simulationsvarianten. Die
Primärenergie-Diagramme verwenden die gespeicherten virtuellen Summen und den
gespeicherten Saldo jeweils mit der Vorzeichenkonvention des Exports.

In [5]:
data = reporting.variables(selected, "QH", envelope, language=LANGUAGE)
display(plot.distribution(**data, variable="QH", by=envelope))

if "QC" in selected and selected["QC"].notna().any():
    data = reporting.variables(selected, "QC", envelope, language=LANGUAGE)
    display(plot.distribution(**data, variable="QC", by=envelope, kind="violin"))

color = heating if heating in selected and selected[heating].notna().all() else envelope
columns = ["PEI_virtual_demand", "PEI_virtual_supply"]
if set(columns).issubset(selected):
    energy = selected.dropna(subset=columns)
    if not energy.empty:
        display(reporting.primary_energy_scatter_plot(energy, color=color, language=LANGUAGE))

columns = ["FSI", "PEI_balance"]
if set(columns).issubset(selected):
    density = selected.dropna(subset=columns)
    if not density.empty:
        display(reporting.density_balance_scatter_plot(density, color=color, language=LANGUAGE))

## Bilanzbeiträge ausgewählter Hüllvarianten vergleichen

Das Beispiel wählt die Gruppe mit den meisten unterschiedlichen Hüllpresets.
Innerhalb dieser Gruppe sind der Simulationslauf, die übrigen aufgezeichneten Presets,
der PV-Dachflächenanteil und das Wetter gleich, soweit diese Spalten im Export
vorhanden sind. Weitere Eingabegrößen können variieren. Für die Grafiken wird
pro Hüllpreset ein Treffer ausgewählt, bis zu sechs Varianten.

Für diese Szenarien werden alle Jahreswerte nachgeladen. Die Reportdefinitionen
in PEXL bestimmen die dargestellten Beiträge.

In [6]:
candidates = selected.dropna(subset=[envelope])
if candidates.empty:
    raise ValueError("Für den Bilanzvergleich braucht die Auswahl ein aufgezeichnetes Hüllpreset.")

controls = [name for name in presets if name != envelope]
controls += [name for name in ("PV_roof_share", "weather_name") if name in candidates]
groups = candidates.reset_index().groupby(["batch", *controls], dropna=False, sort=False)
_, group = max(groups, key=lambda item: item[1][envelope].nunique())
comparison = group.drop_duplicates(envelope).head(6).set_index(identity)
batch = comparison.index[0][0]
display(comparison[[geometry, envelope, "QH"]])

names = comparison.index.get_level_values("scenario_name").tolist()
detail = pd.read_parquet(
    parquet_path,
    filters=[("batch", "==", batch), ("scenario_name", "in", names)],
).set_index(identity).loc[comparison.index]

preset_recorded_geometry  ...          QH
batch              scenario_name                                                             ...            
building_sector_v2 EFH_1G|1919-44|WP_Sole|PV30|noFlex|Wien |Leicht                   EFH_1G  ...  416.099140
                   EFH_1G|1945-59|WP_Sole|PV30|noFlex|Wien |Leicht                   EFH_1G  ...  412.841141
                   EFH_1G|1960-79|WP_Sole|PV30|noFlex|Wien |Leicht                   EFH_1G  ...  423.799506
                   EFH_1G|1980-89|WP_Sole|PV30|noFlex|Wien |Leicht                   EFH_1G  ...  221.183579

[4 rows x 3 columns]

In [7]:
for name in ("heat_balance", "final_energy_balance", "primary_energy_balance", "gwp_ee"):
    report = reporting.materialize(detail, name, language=LANGUAGE)
    columns = list(dict.fromkeys(series["var_name"] for series in report["series"]))
    values = report["frame"][columns]
    missing = values.columns[values.isna().any()].tolist()
    if missing:
        print(f"{report['title']}: fehlende Werte in {', '.join(missing)}")
        continue
    display(render(report))

Report heat_balance uses schema 1_13_3; source versions ['1.13.1_dev'] need interpretation with care
Report final_energy_balance uses schema 1_13_3; source versions ['1.13.1_dev'] need interpretation with care
Report primary_energy_balance uses schema 1_13_3; source versions ['1.13.1_dev'] need interpretation with care
Report gwp_ee uses schema 1_13_3; source versions ['1.13.1_dev'] need interpretation with care
